In [1]:
print("Test Run")

Test Run


In [2]:
import os
import re
import json
import uuid
from typing import Dict, Any, List, Tuple
from datetime import datetime

import boto3
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels


d:\Tutorial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
AWS_REGION = "us-east-1"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

# Embedding model (common Titan embeddings)
EMBED_MODEL_ID = "amazon.titan-embed-text-v1"

# LLM model (example: Claude)
LLM_MODEL_ID = "anthropic.claude-3-5-sonnet-20240620-v1:0"


In [3]:
def bedrock_embed(text: str) -> List[float]:
    """
    Returns embedding vector for a given text using Bedrock Titan embeddings.
    """
    text = (text or "").strip()
    if not text:
        return []

    body = json.dumps({"inputText": text})
    resp = bedrock_runtime.invoke_model(
        modelId=EMBED_MODEL_ID,
        body=body,
        accept="application/json",
        
        contentType="application/json",
    )
    data = json.loads(resp["body"].read())
    return data["embedding"]


In [5]:
def bedrock_split_jd(jd_text: str) -> Dict[str, str]:
    """
    Uses an LLM to split JD into:
    - skills_view
    - responsibilities_view
    (optionally profile/education if you want later)
    """
    prompt = f"""
You are given a Job Description (JD).
Task: Extract only what is explicitly present. Do not add new facts.

Return STRICT JSON with keys:
- "skills_view": a clean list-like text of required skills/tools/keywords from the JD
- "responsibilities_view": a clean list-like text of responsibilities/tasks/deliverables from the JD

JD:
\"\"\"{jd_text}\"\"\"
"""

    # Claude-style request format (Bedrock runtime)
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 800,
        "temperature": 0.0,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    resp = bedrock_runtime.invoke_model(
        modelId=LLM_MODEL_ID,
        body=json.dumps(body),
        accept="application/json",
        contentType="application/json",
    )
    data = json.loads(resp["body"].read())

    # Extract Claude response text
    out_text = data["content"][0]["text"].strip()

    # Parse JSON safely
    try:
        out_json = json.loads(out_text)
        return {
            "skills_view": out_json.get("skills_view", ""),
            "responsibilities_view": out_json.get("responsibilities_view", "")
        }
    except Exception:
        # Fallback: if model returns non-json, keep raw
        return {"skills_view": "", "responsibilities_view": jd_text}


In [ ]:
client = QdrantClient(host = 'localhost', port = 6333)

COLLECTION = "candidates"

test_dim = len(bedrock_embed("test dimension"))
print("Embedding dimension:", test_dim)


In [5]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='candidates'), CollectionDescription(name='resume_points'), CollectionDescription(name='day1_semantic_search'), CollectionDescription(name='new_collection'), CollectionDescription(name='resumes'), CollectionDescription(name='resume_points_testing'), CollectionDescription(name='test_hr_resumes'), CollectionDescription(name='candidates_rag_demo'), CollectionDescription(name='resumes_rag'), CollectionDescription(name='first_collection')])

In [5]:
def create_collection_if_not_exists(collection_name: str):
    dim = len(bedrock_embed("dimension check"))

    existing = [c.name for c in client.get_collections().collections]
    if collection_name in existing:
        print(f"Collection '{collection_name}' already exists.")
        return dim

    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "v_profile": qmodels.VectorParams(size=dim, distance=qmodels.Distance.COSINE),
            "v_skills": qmodels.VectorParams(size=dim, distance=qmodels.Distance.COSINE),
            "v_experience": qmodels.VectorParams(size=dim, distance=qmodels.Distance.COSINE),
            "v_education": qmodels.VectorParams(size=dim, distance=qmodels.Distance.COSINE),
        },
        hnsw_config=qmodels.HnswConfigDiff(m=16, ef_construct=256),
    )
    print(f"✅ Created collection '{collection_name}' (dim={dim}) with named vectors + HNSW.")
    return dim

DIM = create_collection_if_not_exists(COLLECTION)

Collection 'candidates' already exists.


In [11]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='resumes'), CollectionDescription(name='candidates'), CollectionDescription(name='first_collection'), CollectionDescription(name='resumes_rag'), CollectionDescription(name='test_hr_resumes')])

In [6]:
def build_profile_text(r: Dict[str, Any]) -> str:
    c = r.get("candidate", {}) or {}
    return "\n".join([
        f"Department: {c.get('department','')}",
        f"Location: {c.get('location','')}",
        f"Total experience: {c.get('total_years_experience','')} years",
        f"Summary: {c.get('summary','')}",
    ]).strip()

def build_skills_text(r: Dict[str, Any]) -> str:
    s = r.get("skills", {}) or {}

    def join_list(x):
        x = x or []
        return "; ".join([str(i).strip() for i in x if str(i).strip()])

    return "\n".join([
        f"Core skills: {join_list(s.get('core'))}",
        f"Technical skills: {join_list(s.get('technical'))}",
        f"Tools: {join_list(s.get('tools'))}",
        # f"Soft skills: {join_list(s.get('soft'))}",
        # f"Domains: {join_list(s.get('domains'))}",
    ]).strip()

def build_experience_text(r: Dict[str, Any]) -> str:
    exp = r.get("experience", []) or []
    parts = []
    for e in exp:
        bullets = e.get("bullets", []) or []
        bullet_text = " | ".join(bullets[:12])  # limit for readability
        parts.append(
            f"Role: {e.get('job_title','')} | Company: {e.get('company','')} | "
            f"Dates: {e.get('start_date','')} to {e.get('end_date') or 'Present'} | "
            f"Highlights: {bullet_text}"
        )
    return "\n".join(parts).strip()

def build_education_text(r: Dict[str, Any]) -> str:
    edu = r.get("education", []) or []
    cert = r.get("certifications", []) or []

    lines = ["Education:"]
    for e in edu:
        lines.append(
            f"{e.get('degree','')} {e.get('field_of_study','') or ''}, {e.get('institution','')}, "
            f"{e.get('end_date','') or ''} (Grade: {e.get('grade_or_gpa','')})"
        )

    lines.append("Certifications:")
    if not cert:
        lines.append("None")
    else:
        for c in cert:
            lines.append(str(c))

    return "\n".join(lines).strip()


In [7]:
def normalize_list(xs: List[str]) -> List[str]:
    out = []
    for x in xs or []:
        x = str(x).strip().lower()
        if x:
            out.append(x)
    return sorted(list(set(out)))

def build_payload(r: Dict[str, Any]) -> Dict[str, Any]:
    c = r.get("candidate", {}) or {}
    s = r.get("skills", {}) or {}
    exp = r.get("experience", []) or []
    edu = r.get("education", []) or []

    skills_flat = normalize_list(
        (s.get("core") or []) + (s.get("technical") or []) + (s.get("tools") or []) +
        (s.get("soft") or []) + (s.get("domains") or [])
    )
    companies = normalize_list([e.get("company", "") for e in exp])
    job_titles = normalize_list([e.get("job_title", "") for e in exp])
    degrees = normalize_list([e.get("degree", "") for e in edu])

    return {
        "full_name": c.get("full_name"),
        "email": c.get("email"),
        "phone": c.get("phone"),
        "location": c.get("location"),
        "department": c.get("department"),
        "total_years_experience": c.get("total_years_experience"),
        "skills_flat": skills_flat,
        "companies": companies,
        "job_titles": job_titles,
        "degrees": degrees,

        # Full evidence sections:
        "candidate_json": c,
        "skills_json": s,
        "experience_json": exp,
        "education_json": edu,
        "certifications_json": r.get("certifications", []),
        "metadata": r.get("metadata", {}),
    }

def stable_point_id(resume_json: Dict[str, Any]) -> str:
    """
    Qdrant Local requires UUID point ids.
    This generates a stable UUID from email/phone/full_name (in that order).
    """
    c = resume_json.get("candidate", {}) or {}
    key = c.get("email") or c.get("phone") or c.get("full_name") or str(uuid.uuid4())
    # uuid5 gives deterministic UUID for the same key
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, str(key).strip().lower()))


In [8]:
# Code to test points and associated payload

# try:
#     scroll_result, next_offset = client.scroll(
#         collection_name=COLLECTION,
#         limit=10,  # Retrieve a maximum of 10 points per request
#         with_payload=True,  # Set to True to get the payload
#         with_vectors=False, # Set to True to get the vectors, False to omit them
#     )

#     # 3. Iterate over the retrieved points
#     print(f"Retrieved {len(scroll_result)} points:")
#     for point in scroll_result:
#         print(f"Point ID: {point.id}")
#         for key, value in point.payload.items():
#             print(f"  {key}: {value}")
#         # print(f"Payload: {point.payload}")
#         # print(f"Vector: {point.vector}") # Uncomment if with_vectors=True
#         print("-" * 20)

#     # 4. Handle pagination if there are more points
#     if next_offset:
#         print(f"More points available. Use offset {next_offset} for the next page.")

# except Exception as e:
#     print(f"An error occurred: {e}")

In [10]:
def upsert_resume(resume_json: Dict[str, Any]) -> str:
    pid = stable_point_id(resume_json)

    profile_text = build_profile_text(resume_json)
    skills_text = build_skills_text(resume_json)
    experience_text = build_experience_text(resume_json)
    education_text = build_education_text(resume_json)

    vectors = {
        "v_profile": bedrock_embed(profile_text),
        "v_skills": bedrock_embed(skills_text),
        "v_experience": bedrock_embed(experience_text),
        "v_education": bedrock_embed(education_text),
    }

    payload = build_payload(resume_json)

    client.upsert(
        collection_name=COLLECTION,
        points=[
            qmodels.PointStruct(
                id=pid,          # ✅ UUID string
                vector=vectors,  # ✅ named vectors
                payload=payload
            )
        ],
    )

    return pid


In [16]:
with open(r"Resume_Json\Vikas Verma.json", "r", encoding="utf-8") as f:
    resume_json = json.load(f)

pid = upsert_resume(resume_json)
print("✅ Upserted:", pid)


✅ Upserted: d35410d4-877c-5e5d-88d3-d27792cffcc3


In [11]:
import glob
from pathlib import Path

In [12]:
def batch_ingest_folder(folder_path: str) -> Dict[str, Any]:
    """
    Reads all .json resumes from folder_path and upserts into Qdrant.
    Returns stats and list of failed files.
    """
    folder = Path(folder_path)
    files = sorted(glob.glob(str(folder / "*.json")))

    ok, failed = 0, []
    ids = []

    for fp in files:
        try:
            with open(fp, "r", encoding="utf-8") as f:
                rj = json.load(f)

            pid = upsert_resume(rj)
            ids.append(pid)
            ok += 1

        except Exception as e:
            failed.append({"file": fp, "error": str(e)})

    return {
        "folder": str(folder),
        "total_files": len(files),
        "ingested": ok,
        "failed": len(failed),
        "failed_files": failed[:10],   # show first 10 failures
        "point_ids_sample": ids[:5]
    }

# Example:
stats = batch_ingest_folder(r"New_Resume_Json")
print(stats)

{'folder': 'New_Resume_Json', 'total_files': 5, 'ingested': 5, 'failed': 0, 'failed_files': [], 'point_ids_sample': ['c48b4784-68e1-5585-ba11-d91c077ba60d', 'fc9d744b-0a14-5c51-9691-1da2fab8176f', 'e9ad7058-1bd0-5b94-9e07-88190a925dd4', '46611442-c74e-5803-b8f7-ffe23480d4cf', '14d777e1-107c-52a6-8826-4bc36d61e791']}


In [19]:
def bedrock_split_jd_4views(jd_text: str) -> Dict[str, str]:
    """
    Uses Bedrock LLM to split JD into 4 semantic views aligned with your vectors.
    Extract-only, no new facts.
    """
    prompt = f"""
You are given a Job Description (JD).
Task: Extract ONLY what is explicitly present. Do NOT add any new facts.

Return STRICT JSON with keys:
- "profile_view": role title/seniority/location/department-type info if present
- "skills_view": required/preferred skills/tools/keywords
- "responsibilities_view": responsibilities/tasks/deliverables
- "education_view": degrees/certifications if present

JD:
\"\"\"{jd_text}\"\"\"
"""

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 900,
        "temperature": 0.0,
        "messages": [{"role": "user", "content": prompt}]
    }

    resp = bedrock_runtime.invoke_model(
        modelId=LLM_MODEL_ID,
        body=json.dumps(body),
        accept="application/json",
        contentType="application/json",
    )
    data = json.loads(resp["body"].read())
    out_text = data["content"][0]["text"].strip()

    try:
        out_json = json.loads(out_text)
        return {
            "profile_view": out_json.get("profile_view", ""),
            "skills_view": out_json.get("skills_view", ""),
            "responsibilities_view": out_json.get("responsibilities_view", ""),
            "education_view": out_json.get("education_view", ""),
        }
    except Exception:
        # Fallback if model doesn't return valid JSON
        return {
            "profile_view": "",
            "skills_view": "",
            "responsibilities_view": jd_text,
            "education_view": ""
        }


In [22]:
def safe_text(x) -> str:
    """
    Ensures x is a clean string.
    Handles dict / list / None safely.
    """
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    # If LLM returned dict/list accidentally, serialize it
    return json.dumps(x, ensure_ascii=False).strip()


In [ ]:
def search_candidates_from_jd(
    jd_text: str,
    top_k: int = 10,
    per_vector_limit: int = 80,
    use_profile: bool = True,
    use_education: bool = False,
):
    """
    Multi-vector JD-driven search (SAFE VERSION).
    Works correctly with Qdrant Local (:memory:).
    """

    views = bedrock_split_jd_4views(jd_text)

    # 🔒 SAFE normalization (THIS FIXES YOUR ERROR)
    skills_q_text = safe_text(views.get("skills_view")) or jd_text
    exp_q_text    = safe_text(views.get("responsibilities_view")) or jd_text
    prof_q_text   = safe_text(views.get("profile_view")) or jd_text
    edu_q_text    = safe_text(views.get("education_view")) or jd_text

    # embeddings
    q_skills = bedrock_embed(skills_q_text)
    q_exp    = bedrock_embed(exp_q_text)
    q_prof   = bedrock_embed(prof_q_text) if use_profile else None
    q_edu    = bedrock_embed(edu_q_text)  if use_education else None

    # weights (screening-optimized)
    w_exp, w_sk = 0.60, 0.40
    w_prof, w_edu = 0.10, 0.10

    # search
    res_exp = client.search(
        collection_name=COLLECTION,
        query_vector=("v_experience", q_exp),
        limit=per_vector_limit,
        with_payload=True,
    )

    res_sk = client.search(
        collection_name=COLLECTION,
        query_vector=("v_skills", q_skills),
        limit=per_vector_limit,
        with_payload=True,
    )

    res_prof = []
    if use_profile:
        res_prof = client.search(
            collection_name=COLLECTION,
            query_vector=("v_profile", q_prof),
            limit=per_vector_limit,
            with_payload=True,
        )

    res_edu = []
    if use_education:
        res_edu = client.search(
            collection_name=COLLECTION,
            query_vector=("v_education", q_edu),
            limit=per_vector_limit,
            with_payload=True,
        )

    # 🔗 fuse scores
    fused = {}

    def add_results(results, weight, key):
        for r in results:
            pid = str(r.id)
            if pid not in fused:
                fused[pid] = {
                    "id": pid,
                    "score": 0.0,
                    "payload": r.payload,
                    "scores": {}
                }
            fused[pid]["score"] += weight * float(r.score)
            fused[pid]["scores"][key] = float(r.score)

    add_results(res_exp, w_exp, "experience_score")
    add_results(res_sk,  w_sk,  "skills_score")
    if use_profile:
        add_results(res_prof, w_prof, "profile_score")
    if use_education:
        add_results(res_edu, w_edu, "education_score")

    ranked = sorted(fused.values(), key=lambda x: x["score"], reverse=True)[:top_k]

    # attach views for debugging / transparency
    for r in ranked:
        r["jd_views"] = views

    return ranked


In [ ]:
def simple_tokenize(text: str) -> List[str]:
    toks = re.findall(r"[A-Za-z0-9\+\#\.\-]{2,}", (text or "").lower())
    stop = {
        "and","or","the","to","of","in","for","with","a","an","is","are","on","as","at","by","be",
        "we","you","our","will","shall","from","this","that","it"
    }
    toks = [t for t in toks if t not in stop]
    return toks

def extract_evidence(jd_text: str, payload: Dict[str, Any], max_bullets: int = 6) -> Dict[str, Any]:
    jd_tokens = set(simple_tokenize(jd_text))

    skills_flat = payload.get("skills_flat", []) or []
    matched_skills = [s for s in skills_flat if s in jd_tokens]

    matched_bullets = []
    exp_list = payload.get("experience_json", []) or []
    for e in exp_list:
        for b in (e.get("bullets") or []):
            b_low = b.lower()
            hit = any(tok in b_low for tok in list(jd_tokens)[:200])
            if hit:
                matched_bullets.append({
                    "company": e.get("company"),
                    "job_title": e.get("job_title"),
                    "bullet": b
                })
            if len(matched_bullets) >= max_bullets:
                break
        if len(matched_bullets) >= max_bullets:
            break

    return {
        "matched_skills": matched_skills[:15],
        "matched_experience_bullets": matched_bullets,
    }


In [25]:
def print_topk_results(results: List[Dict[str, Any]], jd_text: str):
    for r in results:
        p = r["payload"]
        ev = extract_evidence(jd_text, p)

        print("\n" + "="*100)
        print(f"Rank: {results.index(r)+1} | Final Score: {r['score']:.4f}")
        print(f"Name: {p.get('full_name')} | Dept: {p.get('department')} | Exp: {p.get('total_years_experience')} years")
        print(f"Location: {p.get('location')}")
        print(f"Email: {p.get('email')} | Phone: {p.get('phone')}")

        print("\nScores breakdown:", r.get("scores", {}))

        print("\nMatched skills:")
        if ev["matched_skills"]:
            print(" - " + ", ".join(ev["matched_skills"]))
        else:
            print(" - (No exact skill keyword overlaps found)")

        print("\nTop experience evidence bullets:")
        if ev["matched_experience_bullets"]:
            for b in ev["matched_experience_bullets"]:
                print(f" - {b['company']} | {b['job_title']} -> {b['bullet']}")
        else:
            print(" - (No strong bullet evidence found)")


In [27]:
jd_procurement = """
Job Title: Strategic Sourcing / Procurement Manager

Role Overview:
We are looking for a procurement professional to manage RFQs, costing and negotiation with suppliers.
The role requires coordination with cross-functional teams, supplier development, and handling new product development sourcing.

Responsibilities:
- Handle RFQs end-to-end and convert RFQs to purchase orders
- Supplier development, negotiations, and cost optimization
- Track raw material price changes and update pricing accordingly
- Work on ECNs and update part pricing as required
- Strong communication with export / international customers and stakeholders
- Use SAP for procurement processes

Requirements:
- 8+ years experience in procurement / strategic sourcing
- Strong knowledge of sheet metal / fabrication / automotive components
- Must have negotiation and costing experience
- Preferred: SAP, AutoCAD

Location: India
"""

# results = search_candidates_from_jd(jd_procurement, top_k=5, use_profile=True, use_education=False)
# print_topk_results(results, jd_procurement)


In [ ]:
import qdrant_client
print("qdrant-client version:", qdrant_client.__version__)
print("Has search:", hasattr(client, "search"))
print("Has query_points:", hasattr(client, "query_points"))
print("Has query:", hasattr(client, "query"))


In [28]:
def qdrant_vector_query(collection_name: str, vector_name: str, vector: List[float], limit: int = 50):
    """
    Version-safe Qdrant query that works whether client exposes:
    - search()
    - query_points()
    - query()
    """
    # Newer qdrant-client
    if hasattr(client, "query_points"):
        res = client.query_points(
            collection_name=collection_name,
            query=vector,                 # vector itself
            using=vector_name,            # named vector
            limit=limit,
            with_payload=True
        )
        # query_points returns an object with .points
        return res.points

    # Older qdrant-client
    if hasattr(client, "search"):
        return client.search(
            collection_name=collection_name,
            query_vector=(vector_name, vector),
            limit=limit,
            with_payload=True
        )

    # Some versions have query()
    if hasattr(client, "query"):
        return client.query(
            collection_name=collection_name,
            query_vector=(vector_name, vector),
            limit=limit,
            with_payload=True
        )

    raise RuntimeError("Your qdrant-client does not support search/query_points/query. Please upgrade qdrant-client.")


In [29]:
def search_candidates_from_jd(
    jd_text: str,
    top_k: int = 10,
    per_vector_limit: int = 80,
    use_profile: bool = True,
    use_education: bool = False,
):
    views = bedrock_split_jd_4views(jd_text)

    # Safe view normalization
    skills_q_text = safe_text(views.get("skills_view")) or jd_text
    exp_q_text    = safe_text(views.get("responsibilities_view")) or jd_text
    prof_q_text   = safe_text(views.get("profile_view")) or jd_text
    edu_q_text    = safe_text(views.get("education_view")) or jd_text

    # Embed
    q_skills = bedrock_embed(skills_q_text)
    q_exp    = bedrock_embed(exp_q_text)
    q_prof   = bedrock_embed(prof_q_text) if use_profile else None
    q_edu    = bedrock_embed(edu_q_text)  if use_education else None

    # Weights
    w_exp, w_sk = 0.60, 0.40
    w_prof, w_edu = 0.10, 0.10

    # Query using version-safe wrapper ✅
    res_exp = qdrant_vector_query(COLLECTION, "v_experience", q_exp, limit=per_vector_limit)
    res_sk  = qdrant_vector_query(COLLECTION, "v_skills", q_skills, limit=per_vector_limit)

    res_prof = []
    if use_profile:
        res_prof = qdrant_vector_query(COLLECTION, "v_profile", q_prof, limit=per_vector_limit)

    res_edu = []
    if use_education:
        res_edu = qdrant_vector_query(COLLECTION, "v_education", q_edu, limit=per_vector_limit)

    # Fuse
    fused = {}

    def add_results(results, weight, key):
        for r in results:
            pid = str(r.id)
            if pid not in fused:
                fused[pid] = {"id": pid, "score": 0.0, "payload": r.payload, "scores": {}}
            fused[pid]["score"] += weight * float(r.score)
            fused[pid]["scores"][key] = float(r.score)

    add_results(res_exp, w_exp, "experience_score")
    add_results(res_sk,  w_sk,  "skills_score")
    if use_profile:
        add_results(res_prof, w_prof, "profile_score")
    if use_education:
        add_results(res_edu, w_edu, "education_score")

    ranked = sorted(fused.values(), key=lambda x: x["score"], reverse=True)[:top_k]

    for r in ranked:
        r["jd_views"] = views

    return ranked


In [30]:
jd_design = """
Job Title: Mechanical Design Engineer (Sheet Metal / Tooling)

Responsibilities:
- Create mechanical drawings and documentation for sheet metal components
- Design tooling such as punching, blanking, bending dies and fixtures
- Work closely with production, quality and tool room for product development
- Support new product development and coordinate across departments

Requirements:
- B.E / Diploma in Mechanical Engineering
- 4+ years experience in mechanical design for sheet metal components
- Must have AutoCAD
- Preferred: CATIA V5, Pro-E
"""


In [32]:
pjd = """ 
Job Title: Strategic Sourcing / Procurement Manager

Role Overview:
We are looking for a procurement professional to manage RFQs, costing and negotiation with suppliers.
The role requires coordination with cross-functional teams, supplier development, and handling new product development sourcing.

Responsibilities:
- Handle RFQs end-to-end and convert RFQs to purchase orders
- Supplier development, negotiations, and cost optimization
- Track raw material price changes and update pricing accordingly
- Work on ECNs and update part pricing as required
- Strong communication with export / international customers and stakeholders
- Use SAP for procurement processes

Requirements:
- 8+ years experience in procurement / strategic sourcing
- Strong knowledge of sheet metal / fabrication / automotive components
- Must have negotiation and costing experience
- Preferred: SAP, AutoCAD

Location: India
"""

In [33]:
results = search_candidates_from_jd(pjd, top_k=5, use_profile=True, use_education=False)
print_topk_results(results, pjd)


Rank: 1 | Final Score: 0.7782
Name: Aseem Jain | Dept: CD | Exp: 18 years
Location: Antriksh heights , Sec 84,Gurgaon.
Email: er.jainaseem@gmail.com | Phone: +91 9711869786

Scores breakdown: {'experience_score': 0.72928905, 'skills_score': 0.6692693, 'profile_score': 0.72885466}

Matched skills:
 - automotive, communication, negotiation, sourcing

Top experience evidence bullets:
 - Continental Automotive Components (India) Pvt Ltd. | Manager – Supply Chain Management (Advance SCM & Vendor order management) -> Order Management to ensure components/parts (raw material) availability as per planning.
 - Continental Automotive Components (India) Pvt Ltd. | Manager – Supply Chain Management (Advance SCM & Vendor order management) -> Ensuring purchase at specified quality & timely delivery as per requirement schedule.
 - Continental Automotive Components (India) Pvt Ltd. | Manager – Supply Chain Management (Advance SCM & Vendor order management) -> Coordination between various Departments 

In [13]:
# AWS_REGION = "us-east-1"
# EMBED_MODEL_ID = "amazon.titan-embed-text-v1"
# LLM_MODEL_ID   = "anthropic.claude-3-5-sonnet-20240620-v1:0"

In [14]:
#hi you are qdrant developer and your task is to all three isolated files into one, here i have builled a resume screening system and the flow of system is as initially I am parsing resume pdf's using llm and converting into json implemented in resume_parsing.py file and then indexing these json files into qdrant implemented in indexing.txt file and finally screening resume based on jd in new_app.py. Here i want from you is that as of know if new resume comes then i have to manually parse it index it then only i can screen that resumes but now you implement this feature from ui user will upload resumes from ui then give a button like process resumes then from ui make parsing --> indexing -->screening. Please update the code and give me a final code where i can use these features without changing the fucntionalities and ui you can give upload resumes in the left sidebar above vector settings. THe s=final code must be fast so always use best practices.